In [1]:
import pandas as pd
import folium

In [2]:
df_stations = pd.read_csv('Station_Details_Historical.csv')
df_aqi = pd.read_csv('AQI_Category_Historical.csv')

In [3]:
df_merged = df_aqi.merge(df_stations, on='Station_id', how='inner')
df_merged['timestamp'] = pd.to_datetime(df_merged['timestamp'])
df_recent = df_merged.sort_values('timestamp', ascending=False).drop_duplicates(subset=['Station_id'])
df_recent[['Latitude', 'Longitude']] = df_recent['Latitude_Longitude'].str.split(', ', expand=True).astype(float)

In [4]:
map_center = [26.2006, 92.9376]
aqi_map = folium.Map(location=map_center, zoom_start=6)

for _, row in df_recent.iterrows():
    if row['category_name'] == 'Good':
        color = 'darkgreen'
    elif row['category_name'] == 'Satisfactory':
        color = 'lightgreen'
    elif row['category_name'] == 'Moderate':
        color = 'orange'
    elif row['category_name'] == 'Poor':
        color = 'red'
    elif row['category_name'] == 'Very Poor':
        color = 'purple'
    else:
        color = 'darkred'
    
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=8,
        popup=f"City: {row['City']}<br>AQI: {row['Exact_AQI']}<br>Category: {row['category_name']}",
        tooltip=f"{row['City']} - AQI: {row['Exact_AQI']}",
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8
    ).add_to(aqi_map)

aqi_map